# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [52]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [53]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [54]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [55]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [56]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [57]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [58]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [59]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [60]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [61]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," which is mentioned twice. Other domains such as "Creative / Design / Media," "Developer Tools / DevEx," "Productivity Assistants," and "E‑commerce / Marketplaces" are also represented but less frequently. Based on this dataset, "Healthcare / MedTech" is the most common project domain.'

In [62]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project titled "MediMind 17" is focused on security, involving a medical imaging solution that improves early diagnosis through vision transformers.'

In [63]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had generally positive comments about the fintech projects. They considered several projects to be comprehensive, technically mature, promising, well-executed, and with impactful real-world applications. For example, one project was described as having a "comprehensive and technically mature approach," another as a "clever solution with measurable environmental benefit," and others as "promising," "technically ambitious and well-executed," and "solid work with impressive real-world impact." The scores ranged from 76 to 94, reflecting favorable evaluations overall.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [64]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [65]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [66]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Finance / FinTech," as it is mentioned multiple times among the projects.'

In [67]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project "SecureNest 49" is a document summarization and retrieval system designed for enterprise knowledge bases, which falls under the categories of E-commerce/Marketplaces and Legal/Compliance.'

In [68]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges commented that the fintech project, "SynthMind," had a conceptually strong approach, but their feedback noted that the results needed more benchmarking.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

It uses exact string match retrieval and no embeddings. It will find documents with the exact query with high precision. This is useful for use cases when the exact code/string/sku is necessary and no conceptual or semantic context is necessarily needed. For example, when you are searching for an identifier like an error code, a sku number, or an order ID, exact phrases like legal or medical terminologym or any rare terms like scientific names or domain specific terminology.

BM25 will be better when you need information about a specific thing rather than similar things.



## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [69]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [70]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [71]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Healthcare / MedTech," which appears in two of the sample entries. The other mentioned domain, "Security," appears once in the sample. Based on this limited data, it seems that "Healthcare / MedTech" may be the more prevalent domain. However, since the dataset is small and only a few entries are shown, I cannot definitively determine the most common project domain overall.'

In [72]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases directly related to security mentioned. The projects described focus on federated learning to improve privacy in healthcare applications, but no explicit mention of security use cases is given.'

In [73]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments about the fintech projects highlighted their strengths. Specifically, for the project "Pathfinder 27" in the Finance / FinTech domain, the judges praised the "excellent code quality and use of open-source libraries" and gave it a high score of 9.8. There are no particular negative comments about the fintech projects, indicating a positive evaluation overall.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [74]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [75]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [76]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," as it is mentioned multiple times across various projects.'

In [77]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, there is a project called "AetherCompute," which involves an AI model compression suite enabling on-device reasoning for IoT sensors, and another called "EchoLens," which is a hardware-aware model quantization benchmark suite. Additionally, "SecureNest" appears in two entries: one focused on legal and compliance aspects of a hardware-aware model quantization benchmark suite, and another on an enterprise knowledge base system. The mention of "SecureNest" in the context of legal and compliance suggests a focus on security and data privacy considerations.'

In [78]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive comments about the fintech projects. For example, one project described as a hardware-aware model quantization benchmark suite received comments like "Solid work with impressive real-world impact," while another noted as conceptually strong with results needing more benchmarking said, "Conceptually strong but results need more benchmarking." Overall, the judges recognized the value and strong potential of the fintech-related projects, highlighting their robustness, technical maturity, and impact, though some projects also received suggestions for further validation or benchmarking.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

The user might ask a question but the relevant information is worded differently. If we generate multiple reformulations, we can create different versions of ther user query that target different aspects of a similar topic, so that the RAG is more likely to return relevant results. If the user asks about how to change the password but the document talks about updating account information and security, without reformulation any password information may be lost, but with reformulation there is a higher chance a generated query could mention account updates.
You can then rank the results of the different queries, and share the top result(s).


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [79]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [80]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [81]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [82]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [83]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [84]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the project domains mentioned include Security, Creative / Design / Media, Productivity Assistants, and Healthcare / MedTech. Since only a few projects are shown, it's not definitive, but among these, Healthcare / MedTech appears twice, which suggests it might be a common domain in this sample. However, without the full dataset, I cannot determine the most common project domain with certainty."

In [85]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no explicit use cases related to security. The projects mentioned primarily focus on federated learning for improving privacy in healthcare applications, but they do not specifically address security measures or use cases.'

In [86]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects, describing them as technically ambitious, well-executed, comprehensive, and with measurable or impressive real-world impact.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [87]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [88]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [89]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "E‑commerce / Marketplaces," which is mentioned multiple times in the dataset.'

In [90]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. The project titled "MediMind 17" falls under the Security domain and involves a medical imaging solution that aims to improve early diagnosis through vision transformers.'

In [91]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments about the fintech projects were generally positive. For example, the project "SynthMind," which is an AI-powered platform optimizing logistics routes for sustainability in the Finance / FinTech domain, was rated with a high score of 87 and received a judge score of 9.6, with the comment that it was "Conceptually strong but results need more benchmarking." \n\nAdditionally, "PulseAI," another fintech-related project focused on an adaptive fine-tuning pipeline for multilingual reasoning models, scored 95 with a judge score of 8.0, described as "Technically ambitious and well-executed." \n\nOverall, judges acknowledged the strengths and innovative aspects of these fintech projects, noting their technical ambition and conceptual strength, though some suggested that further benchmarking or stronger evaluation metrics could enhance their impact.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [92]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [93]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [94]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [95]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [96]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [97]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice.'

In [98]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security in the provided context. Specifically, the projects "SynthMind" and "BioForge" are associated with security. "SynthMind" involves a medical imaging solution aimed at early diagnosis, but it\'s also listed under the secondary domain of Security. "BioForge" is explicitly categorized under the Security domain and focuses on a medical imaging solution. Additionally, "SecureNest" is another project directly under security, involving a low-latency inference system for autonomous systems.'

In [99]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive comments about the fintech projects. For example, they described "TrendLens 19" as "technically ambitious and well-executed," and "WealthifyAI 16" as having a "comprehensive and technically mature approach." Additionally, "AutoMate 5" was noted for being "a forward-looking idea with solid supporting data," indicating confidence in the potential and quality of these fintech-related projects.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

If sentences are short and repetitive (similar words but different context), that could mean:
-  Over chunking could happen to keep context, but we don't capture the holistic informtion. For example, if steps to update a password are in an FAQ, but the steps require information that is not provided in that specific chunk, there is a chance the response is factual but not helpful by itself. 
- We can lose context because information close in proximity is lumped together but the actual context differs, so the LLM may not accurately parse that information. For example multiple FAQ sections start with a "How do I...", the actual question after might differ but the start is the same so semantic chunking might not work well.

For an FAQ, we could adjust the algorithm in to group all questions and answers together to not lose any context. Generally, we could figure out if there is some pattern that is being followed and update our chunking algorithm to follow that pattern. We could also add in min/max boundaries so that we can group together more information than would normally just be in 1 chunk.
We could also adjust the similarity threshold and be more lenient if the content is already somewhat repetitive.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [101]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [ ]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10, with_debugging_logs=False)

/Users/keertanachandar/Public/Github/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/keertanachandar/Public/Github/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/keertanachandar/Public/Github/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

...

Property 'summary_embedding' already exists in node '92b392'. Skipping!
Property 'summary_embedding' already exists in node '9e185d'. Skipping!
Property 'summary_embedding' already exists in node 'a99be8'. Skipping!
Property 'summary_embedding' already exists in node 'a6e927'. Skipping!
Property 'summary_embedding' already exists in node 'df97cb'. Skipping!
Property 'summary_embedding' already exists in node '99a8ee'. Skipping!
Property 'summary_embedding' already exists in node '75b87a'. Skipping!
Property 'summary_embedding' already exists in node '1e70fa'. Skipping!
Property 'summary_embedding' already exists in node '154782'. Skipping!
Property 'summary_embedding' already exists in node 'fb1b1b'. Skipping!
Property 'summary_embedding' already exists in node '5de4cd'. Skipping!
Property 'summary_embedding' already exists in node 'cda87b'. Skipping!
Property 'summary_embedding' already exists in node '02d149'. Skipping!
Property 'summary_embedding' already exists in node '5b9511'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [104]:
# Convert dataset to pandas DataFrame for easier viewing
df = dataset.to_pandas()
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nDataFrame info:")
print(df.info())


Dataset shape: (12, 4)

Column names: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   user_input          12 non-null     object
 1   reference_contexts  12 non-null     object
 2   reference           12 non-null     object
 3   synthesizer_name    12 non-null     object
dtypes: object(4)
memory usage: 516.0+ bytes
None


In [105]:
# Display the DataFrame
df


,user_input,reference_contexts,reference,synthesizer_name
0,How do Ling and Imas define and interpret the ...,[Introduction ChatGPT launched in November 202...,"Ling and Imas highlight that ChatGPT, launched...",single_hop_specifc_query_synthesizer
1,How does Claude compare to ChatGPT in terms of...,[Table 1: ChatGPT daily message counts (millio...,"According to the provided context, ChatGPT's d...",single_hop_specifc_query_synthesizer
2,What are nonprofessional occupations and how d...,[Variation by Occupation Figure 23 presents va...,Nonprofessional occupations include administra...,single_hop_specifc_query_synthesizer
3,Whay is Seeking Information in the context of ...,[Conclusion This paper studies the rapid growt...,"In the context of ChatGPT usage, Seeking Infor...",single_hop_specifc_query_synthesizer
4,"How do the differences in usage patterns, espe...",[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The context shows that non-work messages have ...,multi_hop_abstract_query_synthesizer
5,How does variation in ChatGPT usage by occupat...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,"The variation in ChatGPT usage by occupation, ...",multi_hop_abstract_query_synthesizer
6,How does the variation in ChatGPT usage by occ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,"The variation in ChatGPT usage by occupation, ...",multi_hop_abstract_query_synthesizer
7,How does the variation in ChatGPT usage by occ...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,Figure 23 illustrates that ChatGPT usage varie...,multi_hop_abstract_query_synthesizer
8,How does the rapid growth of ChatGPT in the US...,[<1-hop>\n\nConclusion This paper studies the ...,The context indicates that ChatGPT's usage in ...,multi_hop_specific_query_synthesizer
9,Considering the rapid growth of ChatGPT usage ...,[<1-hop>\n\nConclusion This paper studies the ...,The first segment highlights that by July 2025...,multi_hop_specific_query_synthesizer


In [112]:
# Show all generated questions with their types
print("All Generated Test Questions:")
print("=" * 80)
for idx, row in df.iterrows():
    print(f"\nQuestion {idx + 1} [{row['synthesizer_name']}]:")
    print(f"  {row['user_input']}")
    print(f"\nReference Answer:\n{row['reference']}")
    print("\n" + "=" * 80)
print("\n" + "=" * 80)


All Generated Test Questions:

Question 1 [single_hop_specifc_query_synthesizer]:
  How do Ling and Imas define and interpret the significance of ChatGPT's rapid adoption and widespread usage among users, particularly in relation to the growth in message volume and the classification of user interactions as discussed in their study?

Reference Answer:
Ling and Imas highlight that ChatGPT, launched in November 2022, experienced unprecedented growth, with 18 billion messages sent weekly by 700 million users as of July 2025, representing about 10% of the global adult population. They emphasize that this rapid diffusion is unparalleled for a new technology. Their study examines consumer usage patterns, utilizing automated classifiers to categorize messages based on various taxonomies, including whether messages are used for paid work, the topic, and interaction type. They analyze the growth in message volume for both work and non-work purposes and explore differences across populations and

In [118]:
# Create a collection of the seven retriever types.
retrievers = {
    "naive": naive_retriever,
    "bm25": bm25_retriever,
    "compression": compression_retriever,
    "multi_query": multi_query_retriever,
    "parent_document": parent_document_retriever,
    "ensemble": ensemble_retriever,
    "semantic": semantic_retriever,
}

In [119]:
import time
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Create a QA prompt template
qa_prompt_template = """Use the following pieces of context to answer the question at the end. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=qa_prompt_template, input_variables=["context", "question"]
)

def create_qa_chain(retriever, llm):
    """
    Create a RetrievalQA chain that uses the retriever to get documents and LLM to generate answers.
    """
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": QA_PROMPT}
    )

def safe_generate_answer(qa_chain, question, max_retries=3, sleep_time=5):
    """
    Call qa_chain.invoke(question), retrying on exceptions (rate/token limits).
    Returns both the generated answer and the source documents.
    """
    for attempt in range(1, max_retries + 1):
        try:
            result = qa_chain.invoke({"query": question})
            return result["result"], result["source_documents"]
        except Exception as e:
            if attempt == max_retries:
                print(f"Failed after {max_retries} attempts: {e}")
                return None, []
            print(f"Attempt {attempt} failed with error: {e}. Retrying after {sleep_time} seconds...")
            time.sleep(sleep_time)


In [125]:
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
    answer_correctness,
)
from datasets import Dataset

sample_metrics = [answer_relevancy, faithfulness, context_recall, context_precision, answer_correctness]

def evaluate_answer_metrics(question, generated_answer, reference_answer, contexts):
    """
    Evaluates the generated answer using RAGAS metrics.
    Returns a dictionary mapping each metric to a score.
    """
    # Create a dataset in the format RAGAS expects
    eval_data = {
        "question": [question],
        "answer": [generated_answer],
        "contexts": [contexts],
        "ground_truth": [reference_answer]
    }
    
    eval_dataset = Dataset.from_dict(eval_data)
    
    try:
        # Evaluate using RAGAS
        result = evaluate(eval_dataset, metrics=sample_metrics)
        return result
    except Exception as e:
        print(f"Error evaluating metrics: {e}")
        return {metric.name: 0.0 for metric in sample_metrics}

def generate_and_evaluate_all_answers(df, retrievers, llm):
    """
    For each question, use each retriever to generate answers, then compare each to the reference answer via metrics.
    Returns an expanded dataframe or results list.
    """
    results = []
    
    for idx, row in df.iterrows():
        question = row["user_input"]
        reference_answer = row["reference"] if "reference" in row else ""
        
        for retriever_name, retriever in retrievers.items():
            print(f"Processing question {idx+1}/{len(df)} with {retriever_name} retriever...")
            
            # Create QA chain for this retriever
            qa_chain = create_qa_chain(retriever, llm)
            
            # Generate answer and get source documents
            generated_answer, source_docs = safe_generate_answer(qa_chain, question)
            
            if generated_answer is None:
                generated_answer = ""
                source_docs = []
            
            # Extract context from source documents
            contexts = [doc.page_content for doc in source_docs]
            
            # Evaluate metrics
            metric_scores = evaluate_answer_metrics(
                question, 
                generated_answer, 
                reference_answer, 
                contexts
            )
            
            # Store results
            result_row = {
                "question_idx": idx,
                "question": question,
                "retriever": retriever_name,
                "generated_answer": generated_answer,
                "reference_answer": reference_answer,
                "num_contexts": len(contexts)
            }
            
            # Add metric scores
            # RAGAS evaluate() returns a Result object with a .to_pandas() method
            if metric_scores is not None:
                if hasattr(metric_scores, 'to_pandas'):
                    # Convert to pandas and extract first row (since we only evaluated 1 sample)
                    scores_df = metric_scores.to_pandas()
                    for col in scores_df.columns:
                        if col not in ['question', 'answer', 'contexts', 'ground_truth']:
                            result_row[col] = scores_df[col].iloc[0]
                elif isinstance(metric_scores, dict):
                    for key, value in metric_scores.items():
                        if key not in ['question', 'answer', 'contexts', 'ground_truth']:
                            result_row[key] = value
            
            results.append(result_row)
    
    import pandas as pd
    return pd.DataFrame(results)



In [126]:
# Convert Testset to DataFrame first
dataset_df = dataset.to_pandas()

# Generate and evaluate answers using all retrievers
results_df = generate_and_evaluate_all_answers(dataset_df, retrievers, chat_model)
display(results_df.head())

Processing question 1/12 with naive retriever...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Processing question 1/12 with bm25 retriever...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Processing question 1/12 with compression retriever...


...

Processing question 12/12 with ensemble retriever...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Processing question 12/12 with semantic retriever...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

,question_idx,question,retriever,generated_answer,reference_answer,num_contexts,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
0,0,How do Ling and Imas define and interpret the ...,naive,I don't have information indicating that Ling ...,"Ling and Imas highlight that ChatGPT, launched...",10,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information indicating that Ling ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.224180
1,0,How do Ling and Imas define and interpret the ...,bm25,I don't have information on Ling and Imas's sp...,"Ling and Imas highlight that ChatGPT, launched...",4,How do Ling and Imas define and interpret the ...,[An interactive 3D environment for generative ...,I don't have information on Ling and Imas's sp...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.221630
2,0,How do Ling and Imas define and interpret the ...,compression,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",3,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.228956
3,0,How do Ling and Imas define and interpret the ...,multi_query,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",12,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.227106
4,0,How do Ling and Imas define and interpret the ...,parent_document,I don't have information regarding Ling and Im...,"Ling and Imas highlight that ChatGPT, launched...",4,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information regarding Ling and Im...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.221647


In [127]:
# Diagnostic: Check the structure of results_df
print("Results DataFrame shape:", results_df.shape)
print("\nColumn names:")
print(results_df.columns.tolist())
print("\nFirst few rows:")
display(results_df.head())
print("\nMetric columns found:")
metric_cols = [col for col in results_df.columns if col not in [
    "question_idx", "question", "retriever", "generated_answer", "reference_answer", "num_contexts"
]]
print(metric_cols)
print("\nSample of data:")
display(results_df[["question_idx", "retriever"] + metric_cols].head(10))


Results DataFrame shape: (84, 15)

Column names:
['question_idx', 'question', 'retriever', 'generated_answer', 'reference_answer', 'num_contexts', 'user_input', 'retrieved_contexts', 'response', 'reference', 'answer_relevancy', 'faithfulness', 'context_recall', 'context_precision', 'answer_correctness']

First few rows:


,question_idx,question,retriever,generated_answer,reference_answer,num_contexts,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
0,0,How do Ling and Imas define and interpret the ...,naive,I don't have information indicating that Ling ...,"Ling and Imas highlight that ChatGPT, launched...",10,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information indicating that Ling ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.224180
1,0,How do Ling and Imas define and interpret the ...,bm25,I don't have information on Ling and Imas's sp...,"Ling and Imas highlight that ChatGPT, launched...",4,How do Ling and Imas define and interpret the ...,[An interactive 3D environment for generative ...,I don't have information on Ling and Imas's sp...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.221630
2,0,How do Ling and Imas define and interpret the ...,compression,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",3,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.228956
3,0,How do Ling and Imas define and interpret the ...,multi_query,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",12,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.227106
4,0,How do Ling and Imas define and interpret the ...,parent_document,I don't have information regarding Ling and Im...,"Ling and Imas highlight that ChatGPT, launched...",4,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information regarding Ling and Im...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.0,0.0,0.0,0.221647



Metric columns found:
['user_input', 'retrieved_contexts', 'response', 'reference', 'answer_relevancy', 'faithfulness', 'context_recall', 'context_precision', 'answer_correctness']

Sample of data:


,question_idx,retriever,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
0,0,naive,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information indicating that Ling ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.224180
1,0,bm25,How do Ling and Imas define and interpret the ...,[An interactive 3D environment for generative ...,I don't have information on Ling and Imas's sp...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.221630
2,0,compression,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.228956
3,0,multi_query,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.227106
4,0,parent_document,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,I don't have information regarding Ling and Im...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.221647
5,0,ensemble,How do Ling and Imas define and interpret the ...,[A generative model enabling adaptive speech s...,I don't have information on how Ling and Imas ...,"Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.226707
6,0,semantic,How do Ling and Imas define and interpret the ...,[An automated data labeling assistant powered ...,"I'm sorry, but I don't have information on how...","Ling and Imas highlight that ChatGPT, launched...",0.0,1.000000,0.0,0.0,0.222068
7,1,naive,How does Claude compare to ChatGPT in terms of...,[A multimodal captioning system trained on ope...,I don't have information comparing Claude to C...,"According to the provided context, ChatGPT's d...",0.0,0.333333,0.0,0.0,0.388195
8,1,bm25,How does Claude compare to ChatGPT in terms of...,[An interactive 3D environment for generative ...,I don't have information comparing Claude to C...,"According to the provided context, ChatGPT's d...",0.0,0.333333,0.0,0.0,0.399844
9,1,compression,How does Claude compare to ChatGPT in terms of...,[A generative model enabling adaptive speech s...,I don't have information on how Claude compare...,"According to the provided context, ChatGPT's d...",0.0,0.250000,0.0,0.0,0.399693


In [132]:
# Display results: one table per question showing retrievers vs metrics
import numpy as np

# Identify the metric columns (exclude known columns)
metric_cols = [col for col in results_df.columns if col not in [
    "question_idx", "question", "retriever", "generated_answer", "reference_answer", "num_contexts"
]]

if len(metric_cols) == 0:
    print("⚠️ No metric columns found! The RAGAS evaluation may not have returned scores properly.")
    print("Available columns:", results_df.columns.tolist())
else:
    print(f"Found {len(metric_cols)} metric columns: {metric_cols}\n")
    print(f"Creating {results_df['question_idx'].nunique()} tables (one per question)...\n")
    
    # Group by question and create one table per question
    for question_idx in sorted(results_df['question_idx'].unique()):
        question_data = results_df[results_df['question_idx'] == question_idx]
        question_text = question_data['question'].iloc[0]
        
        # Create a table with retrievers as rows and metrics as columns
        retriever_scores = []
        for retriever in question_data['retriever'].unique():
            row_data = question_data[question_data['retriever'] == retriever].iloc[0]
            score_dict = {'Retriever': retriever}
            for metric in metric_cols:
                value = row_data[metric]
                # Convert to float if possible, otherwise keep as is
                try:
                    score_dict[metric] = float(value) if value is not None else 0.0
                except (ValueError, TypeError):
                    score_dict[metric] = 0.0
            retriever_scores.append(score_dict)
        
        import pandas as pd
        table = pd.DataFrame(retriever_scores).set_index('Retriever')
        
        print(f"\n{'='*100}")
        print(f"Question {question_idx + 1}")
        print(f"{'='*100}")
        print(f"Q: {question_text[:150]}{'...' if len(question_text) > 150 else ''}\n")
        
        # Display with styling - format numeric columns only
        try:
            styled_table = table.style.format("{:.4f}").set_caption(f"Question {question_idx + 1}: Retriever Performance")
        except:
            # Fallback if formatting fails
            styled_table = table.style.set_caption(f"Question {question_idx + 1}: Retriever Performance")
        
        display(styled_table)
        print("\n")


Found 9 metric columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'answer_relevancy', 'faithfulness', 'context_recall', 'context_precision', 'answer_correctness']

Creating 12 tables (one per question)...


Question 1
Q: How do Ling and Imas define and interpret the significance of ChatGPT's rapid adoption and widespread usage among users, particularly in relation to t...



,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
Retriever,,,,,,,,,
naive,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2242
bm25,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2216
compression,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2290
multi_query,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2271
parent_document,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2216
ensemble,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2267
semantic,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2221





Question 2
Q: How does Claude compare to ChatGPT in terms of daily message volume and usage categories, especially regarding work-related and non-work-related messa...



,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
Retriever,,,,,,,,,
naive,0.0000,0.0000,0.0000,0.0000,0.0000,0.3333,0.0000,0.0000,0.3882
bm25,0.0000,0.0000,0.0000,0.0000,0.0000,0.3333,0.0000,0.0000,0.3998
compression,0.0000,0.0000,0.0000,0.0000,0.0000,0.2500,0.0000,0.0000,0.3997
multi_query,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1864
parent_document,0.0000,0.0000,0.0000,0.0000,0.0000,0.3333,0.0000,0.0000,0.3897
ensemble,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2245
semantic,0.0000,0.0000,0.0000,0.0000,0.0000,0.3333,0.0000,0.0000,0.3899





Question 3
Q: What are nonprofessional occupations and how do they use ChatGPT compared to other jobs?



...

,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
Retriever,,,,,,,,,
naive,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2251
bm25,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1860
compression,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.0000,0.2077
multi_query,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2280
parent_document,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2242
ensemble,0.0000,0.0000,0.0000,0.0000,0.9406,0.0000,0.0000,0.0000,0.9859
semantic,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.0000,0.2295





Question 11
Q: Whats hapend in novembrr 2022 that made ChatGPT so poplar and how did it grow by novembrr 2022?



,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
Retriever,,,,,,,,,
naive,0.0000,0.0000,0.0000,0.0000,0.9334,0.0000,0.0000,0.0000,0.8001
bm25,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2151
compression,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2154
multi_query,0.0000,0.0000,0.0000,0.0000,0.9357,0.0000,0.0000,0.0000,0.5380
parent_document,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1812
ensemble,0.0000,0.0000,0.0000,0.0000,0.9273,0.0000,0.0000,0.0000,0.7619
semantic,0.0000,0.0000,0.0000,0.0000,0.9140,0.0000,0.0000,0.0000,0.7146





Question 12
Q: Wht is the signifcance of July 2025 in the growth of ChatGPT and how does it relate to the overall usage trends and user demographics?



,user_input,retrieved_contexts,response,reference,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
Retriever,,,,,,,,,
naive,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2248
bm25,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2246
compression,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2244
multi_query,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2248
parent_document,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2237
ensemble,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2248
semantic,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.2245


Given that most of my metrics were 0, we can say that the actual comparison between retrievers is ultimately broken. But what we can compare is the performance in terms of  latency and cost. 


Latency Analysis:
Semantic retriever averaged ~5.1 seconds - consistently fast across all queries
BM25 averaged ~5.2 seconds - extremely stable performance
Multi-query averaged ~31.8 seconds with multiple timeout exceptions (307s, 936s spikes)
Compression averaged ~49.8 seconds with catastrophic timeout (498s on question 5)
Ensemble showed instability with 936-second timeout on question 4

Cost Analysis ($0.59 - gpt-4.1-nano):
Total: 3,155 requests, 1,219 embeddings
BM25 = $0 - no embeddings required (keyword-based)
Naive/Semantic = Low-Moderate - single embedding per query
Multi-query = Very High - generates 3-5 reformulations × embeddings
Compression = Very High - embeddings for compression + retrieval
Ensemble = High - combines multiple retrievers (BM25 + embeddings)

If we just go off these results, then the semantic retriever actually performs the best. But since that one doesn't count, the next 2 runner ups are the BM25 or the naive retriever (just based off of this data). The BM25 was the fastest (tied with Naive), didn't timeout, and doesn't cost anything because you don't need to do any embeddings. The naive retriever is a close second - it tied in terms of speed with BmM25 (0.2 seconds slower), and there is only 1 embedding per query, so it doesn't cost much.

Overall, running this even once took about 45 minutes. If the evaluators were correct, we would have used those metrics to actually compare the retrievers. Whichever had the highest average across all the metrics (with correctness probably weighted a little more), would have been the winner. In order to accurately compare these retrievers, we'd have to redo our indexing, chunking, and test with simpler tools rather than jumping straight into the comparison. Since we know BM25 works the best without any evaluators, we can use that while checking if the rest of our system works properly.
